In [19]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings('ignore')

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

In [23]:
class ConversationChain:
    def __init__(self, llm, k=None):
        self.llm = llm
        self.chat_history = []
        self.k = k  # window size

    def predict(self, input):
        # Apply window memory if k is set
        if self.k:
            memory = self.chat_history[-2*self.k:]
        else:
            memory = self.chat_history

        # Build messages
        messages = memory + [{"role": "user", "content": input}]

        # Call LLM
        response = self.llm.invoke(messages)

        # Save to memory
        self.chat_history.append({"role": "user", "content": input})
        self.chat_history.append({"role": "assistant", "content": response.content})

        return response.content

    @property
    def buffer(self):
        return self.chat_history

In [24]:
conversation = ConversationChain(llm=llm)

print(conversation.predict(input="Hi, my name is Andrew"))
print(conversation.predict(input="What is 1+1?"))
print(conversation.predict(input="What is my name?"))

print(conversation.buffer)

Hello Andrew! How can I assist you today?
1 + 1 equals 2.
Your name is Andrew. How can I help you further?
[{'role': 'user', 'content': 'Hi, my name is Andrew'}, {'role': 'assistant', 'content': 'Hello Andrew! How can I assist you today?'}, {'role': 'user', 'content': 'What is 1+1?'}, {'role': 'assistant', 'content': '1 + 1 equals 2.'}, {'role': 'user', 'content': 'What is my name?'}, {'role': 'assistant', 'content': 'Your name is Andrew. How can I help you further?'}]


In [25]:
conversation = ConversationChain(llm=llm, k=1)

print(conversation.predict(input="Hi, my name is Andrew"))
print(conversation.predict(input="What is 1+1?"))
print(conversation.predict(input="What is my name?"))

Hello Andrew! How can I assist you today?
1 + 1 equals 2.
I don’t have access to your name based on our current conversation. If you’d like, you can tell me your name!


In [26]:
conversation = ConversationChain(llm=llm, k=2)

print(conversation.predict(input="Hi, my name is Andrew"))
print(conversation.predict(input="What is 1+1?"))
print(conversation.predict(input="What is my name?"))

Hello Andrew! How can I assist you today?
1 + 1 equals 2.
Your name is Andrew. How can I help you further?


In [27]:
import tiktoken

def count_tokens(messages, model="gpt-4.1-mini"):
    enc = tiktoken.encoding_for_model(model)
    return sum(len(enc.encode(m["content"])) for m in messages)


class TokenConversationChain(ConversationChain):
    def __init__(self, llm, max_tokens=100):
        super().__init__(llm)
        self.max_tokens = max_tokens

    def get_token_memory(self):
        result = []
        total = 0

        for msg in reversed(self.chat_history):
            tokens = count_tokens([msg])
            if total + tokens > self.max_tokens:
                break
            result.insert(0, msg)
            total += tokens

        return result

    def predict(self, input):
        memory = self.get_token_memory()
        messages = memory + [{"role": "user", "content": input}]
        response = self.llm.invoke(messages)

        self.chat_history.append({"role": "user", "content": input})
        self.chat_history.append({"role": "assistant", "content": response.content})

        return response.content

In [29]:
conversation = TokenConversationChain(llm=llm, max_tokens=50)

print(conversation.predict("Hi, my name is Andrew"))
print(conversation.predict("What is 1+1?"))
print(conversation.predict("What is my name?"))

Hi Andrew! How can I assist you today?
1 + 1 equals 2.
Your name is Andrew. How can I help you further, Andrew?


In [30]:
class SummaryConversationChain(ConversationChain):
    def __init__(self, llm, max_messages=6):
        super().__init__(llm)
        self.max_messages = max_messages

    def summarize(self):
        summary_prompt = [
            {"role": "system", "content": "Summarize the conversation briefly"},
            *self.chat_history
        ]
        response = self.llm.invoke(summary_prompt)
        return response.content

    def predict(self, input):
        if len(self.chat_history) > self.max_messages:
            summary = self.summarize()
            memory = [{"role": "system", "content": f"Summary: {summary}"}]
        else:
            memory = self.chat_history

        messages = memory + [{"role": "user", "content": input}]
        response = self.llm.invoke(messages)

        self.chat_history.append({"role": "user", "content": input})
        self.chat_history.append({"role": "assistant", "content": response.content})

        return response.content

In [31]:
conversation = SummaryConversationChain(llm=llm)

print(conversation.predict("Hi, my name is Andrew"))
print(conversation.predict("I like machine learning"))
print(conversation.predict("What is 1+1?"))
print(conversation.predict("What is my name?"))

Hello Andrew! How can I assist you today?
That's great to hear, Andrew! Machine learning is a fascinating and rapidly evolving field. Are you working on any specific projects or topics within machine learning?
1 + 1 equals 2. If you have any other questions, feel free to ask!
Your name is Andrew. How can I assist you further today?
